In [1]:
import heapq

def read_state(prompt):
    while True:
        try:
            state = tuple(map(int, input(prompt).split()))
            if len(state) == 9 and sorted(state) == list(range(9)):
                return state
        except ValueError:
            pass
        print("Enter numbers 0 to 8 exactly once.")

def heuristic(state, goal):
    goal_positions = {
        tile: divmod(i, 3)
        for i, tile in enumerate(goal)
    }

    distance = 0

    for i, tile in enumerate(state):
        if tile == 0:
            continue

        row, col = divmod(i, 3)
        goal_row, goal_col = goal_positions[tile]

        distance += abs(row - goal_row) + abs(col - goal_col)

    return distance

def get_neighbors(state):
    zero = state.index(0)
    row, col = divmod(zero, 3)
    neighbors = []

    for move, dr, dc in [
        ("Up", -1, 0), ("Down", 1, 0),
        ("Left", 0, -1), ("Right", 0, 1)
    ]:
        nr, nc = row + dr, col + dc

        if 0 <= nr < 3 and 0 <= nc < 3:
            pos = nr * 3 + nc
            new_state = list(state)
            new_state[zero], new_state[pos] = new_state[pos], new_state[zero]
            neighbors.append((tuple(new_state), move))

    return neighbors

def is_solvable(initial, goal):
    def parity(state):
        a = [x for x in state if x != 0]
        return sum(a[i] > a[j]
                   for i in range(len(a))
                   for j in range(i + 1, len(a))) % 2
    return parity(initial) == parity(goal)

def a_star(initial, goal):
    pq = [(heuristic(initial, goal), 0, 0, initial, [])]
    best = {initial: 0}
    counter = 0

    while pq:
        f, g, _, state, path = heapq.heappop(pq)

        if state == goal:
            return g, path

        if g != best.get(state):
            continue

        for neighbor, move in get_neighbors(state):
            new_g = g + 1

            if new_g < best.get(neighbor, float("inf")):
                best[neighbor] = new_g
                counter += 1
                h = heuristic(neighbor, goal)
                heapq.heappush(
                    pq, (new_g + h, new_g, counter,
                         neighbor, path + [move])
                )

    return None, None

print("--- A* Search: Manhattan Distance Heuristic ---")

initial = read_state("Enter initial configuration (0 for blank): ")
goal = read_state("Enter goal configuration (0 for blank): ")

if not is_solvable(initial, goal):
    print("Goal cannot be reached from the initial configuration.")
else:
    depth, moves = a_star(initial, goal)

    if moves is not None:
        print("Initial Configuration:", initial)
        print("Goal reached at depth (g):", depth)
        print("Move Sequence:", moves)

--- A* Search: Manhattan Distance Heuristic ---
Enter initial configuration (0 for blank): 2 8 3 1 6 4 7 0 5
Enter goal configuration (0 for blank): 1 2 3 8 0 4 7 6 5
Initial Configuration: (2, 8, 3, 1, 6, 4, 7, 0, 5)
Goal reached at depth (g): 5
Move Sequence: ['Up', 'Up', 'Left', 'Down', 'Right']
